In [1]:
##
# Measure clustering effect 
# - final function-loop version for the book
#

import pandas as pd
import numpy as np
import scipy as sp
import sklearn
import importlib
import databox as db
import joblib
from hashlib import md5
# import sample_group_stats as sgst
from sklearn.mixture import BayesianGaussianMixture
from sklearn.decomposition import FastICA
import extools as ex
import scipy.stats as st
import scipy.special as sp

OK -- read the ontology db (19571, 4)
Restored 303734 tags and 191240 tokens. Run update_corpus_tagdata() to update.
-- loaded the skipgram model tank/fasttext_8.model --


In [2]:
ica_data_df = pd.read_parquet("data/q_sep/CE-ICA_data_8-dim.pq")

tf_df = pd.read_parquet("data/q_sep/token_features.pq")

ica8 = joblib.load("data/q_sep/CE-ICA_transformer_w_8-dim.pkl")

In [56]:
print("=========================")


feat_freq_df = pd.DataFrame(tf_df.reset_index().drop_duplicates().\
                            groupby(['category','arcpath']).lemma.value_counts())
feat_freq_df = feat_freq_df[feat_freq_df['count']>1].copy()
print("feat_freq_df.shape", feat_freq_df.shape)

print("=========================")


feat_idx = feat_freq_df.index.droplevel(0).unique()

# feat_idx is the n>1 features in the data set, that we evaluate

print("feat_idx: the set of features in the data to evaluate")
print(len(feat_idx))

print("=========================")

feat_vectors =\
  [ex.compute_feat_vec(arcpath,lemma,ica8).round(3) for (arcpath,lemma) in feat_idx]
feat_vectors_df = pd.DataFrame(feat_vectors,dtype=np.float16,index=feat_idx)

print("feat_vectors_df.shape",feat_vectors_df.shape)
feat_tokens = tf_df[['arcpath','lemma']].reset_index()[['token','arcpath','lemma']].\
  drop_duplicates().set_index(['arcpath','lemma'])

print("=========================")

print("check counts: feat_freq_df.groupby('category')")
print([df.sort_values('count').tail() for g,df in feat_freq_df.groupby('category')])

print("=========================")

tf_df = tf_df[(tf_df.reset_index().set_index(['category','arcpath','lemma']).index.isin(feat_freq_df.index))]
tf_df=tf_df[['arcpath','lemma']]
tf_df.head(5)



feat_freq_df.shape (56151, 1)
feat_idx: the set of features in the data to evaluate
45214
feat_vectors_df.shape (45214, 8)
check counts: feat_freq_df.groupby('category')
[                             count
category   arcpath    lemma       
adaptation conj>cc>   and      204
           nmod>case> of       207
           det>       the      222
           case>      of       232
           cop>       be       430,                               count
category arcpath       lemma       
location compound<det> the      177
         conj>cc>      and      194
         amod<         bear     304
         case>         in       311
         det>          the      404,                             count
category   arcpath   lemma       
taxongroup cc>       and      476
           amod>     other    490
           nmod<det> the      494
           case>     of      1219
           det>      the     1533,                               count
category arcpath       lemma       
topic    conj<conj>

arcpath lemma
category   token                    widx                      
adaptation Aar-5e58_102_predators-8 5               obl<  warn
                                    6           obl<obj>    it
                                    7              case>    of
                                    22    appos>conj>cc>   and
           Aar-5e58_110_nocturnal-4 1         nsubj>det>   the

In [45]:
import joblib
import os

n_clust = 10

def get_BGMM(my_cat : str):
    

    my_ica_data = ica_data_df.loc[my_cat]
    
    bgm_pkl_fname = os.path.join('data/q_cluster','BGM_full_covar_'+my_cat+'.pkl')
    print("checking for %s" % bgm_pkl_fname)
    if os.path.exists(bgm_pkl_fname):
        bgm = joblib.load(bgm_pkl_fname)
        print("BGM loaded")
        # print(str(bgm.__dict__))
    else:
    
        bgm = BayesianGaussianMixture(n_components=n_clust,tol=0.001,max_iter=500,
                                              random_state=42,
                                              covariance_type='full')
        bgm.fit(my_ica_data)
        
        joblib.dump(bgm,bgm_pkl_fname)
        print("BGM saved: %s" % bgm_pkl_fname)
        
    return bgm


In [46]:
bgm_ada = get_BGMM('adaptation')

checking for data/q_cluster/BGM_full_covar_adaptation.pkl
BGM loaded


In [317]:
def analyze_category(my_cat, int_struct = False, debug = False):
    # my_cat = 'location'
    # how many picks from each cluster
    global bgm, my_ica_data   
  
    result = {}
    my_tfs = tf_df.loc[my_cat]
    my_ica_data = ica_data_df.loc[my_cat]
    
    bgm = get_BGMM(my_cat)
    
    
    # check fit
    
    print("bgm.predict(my_ica_data)).value_counts()")
    print(pd.Series(bgm.predict(my_ica_data)).value_counts())
    
    my_clust = pd.DataFrame({'cluster':bgm.predict(my_ica_data)},index=my_ica_data.index)
    feat_bag = my_tfs.reset_index()[['arcpath','lemma','token']]
    clust_feats = my_clust.merge(feat_bag,left_index=True,right_on='token').drop_duplicates().groupby(['cluster','arcpath']).lemma.value_counts()

    clust_feats = pd.DataFrame(clust_feats).reset_index().rename(columns={'count':'freq'}).\
        sort_values('freq',ascending=False).set_index(['arcpath','lemma'])
    
    print("clust_feats",clust_feats.shape)
    print(clust_feats.head())
    
    print("=========================")
    
    
    clust_f_df = clust_feats.pivot(columns='cluster').fillna(0).astype(int)
    print("clust_f_df pivot table",clust_f_df.shape)
    
    clust_p_df = pd.DataFrame(np.array(clust_f_df) / (np.array(clust_f_df).sum(axis=0)),index=clust_f_df.index)
    print("clust_p_df is the probability distribution cluster-wise")
    print(clust_p_df.head(5))
    print("=========================")


    print("Elementwise relative entropy by clusters, feature-wise")

    clust_q = clust_p_df.sum(axis=1)
    clust_q /= clust_q.sum()
    clust_el_kldiv = pd.concat([sp.rel_entr(clust_p_df[c],clust_q) for c in clust_p_df.columns],axis=1)
    
    print(clust_el_kldiv.loc[clust_q.sort_values(ascending=False).index[0:5]])
    
    
    print("========== clust_featdx_df ===============")


    clust_featdx_df = pd.concat([pd.DataFrame(clust_el_kldiv[c].sort_values(ascending=False))\
                           .assign(cluster=c).rename(columns={c:'Dx'})
                             for c in clust_el_kldiv.columns]).reset_index().\
                            set_index(['cluster','arcpath','lemma'])


    clust_featdx_df = clust_featdx_df.merge(clust_feats.reset_index().set_index(['cluster','arcpath','lemma'])\
                                .rename(columns={'freq':'N'}),left_index=True,right_index=True)


    clust_featdx_df_short = clust_featdx_df.groupby('cluster').head(5) # table sketch for the book
    print(clust_featdx_df_short.sample(5))
    result['clust_featdx_df_short']=clust_featdx_df_short
    result['clust_featdx_df']=clust_featdx_df

    print("=========================")

    print("pred_feats")
    
    global pred_feats
    
    feats100 = feat_freq_df.loc[my_cat].sort_values('count').tail(100).index    
    
    pred_feats = clust_featdx_df.reset_index().set_index(['arcpath','lemma']).index.unique()
    pred_feats  = pred_feats[pred_feats.isin(feat_idx)].copy()    
    pred_feats_c = pred_feats[pred_feats.isin(feats100)].copy()        
    pred_feats_all = pred_feats
    
    pred_feats = pred_feats_c # cluster prediction
    
    feat_v = feat_vectors_df.loc[pred_feats].sort_index()

    print("feat_v")
    print(feat_v.round(3).head(5))

    mean_norm = (bgm.means_.T/np.sqrt((bgm.means_**2).sum(axis=1))).T.round(2)
    print("mean_norm")
    print(mean_norm)

    cos_predict = pd.DataFrame(np.inner(mean_norm,np.array(feat_v)).T,index=pred_feats)
    print("cos_predict.head(5)")
    print(cos_predict.head(5))
    
    score_predict = pd.DataFrame(bgm.predict_proba(feat_v).round(3),index=pred_feats)
    print("score_predict.head(5)")
    print(score_predict.head(5))

    prob_sheet = clust_p_df.loc[pred_feats]
    print(prob_sheet.head(5))

    cos_pred_stats   = [st.spearmanr(prob_sheet.loc[i],cos_predict.loc[i])   for i in pred_feats]
    score_pred_stats = [st.spearmanr(prob_sheet.loc[i],score_predict.loc[i]) for i in pred_feats]
    
    df = pd.concat([pd.DataFrame(cos_pred_stats,index=pred_feats),
              pd.DataFrame(score_pred_stats,index=pred_feats)],axis=1)
    df.columns=['cos_stat','cos_pvalue','score_stat','score_pvalue']
    
    result["cos_pvalue"]=df.sort_values('cos_pvalue')
    result["score_pvalue"]=df.sort_values('score_pvalue')
    
    
    pred_feats = pred_feats_all
    feat_v = feat_vectors_df.loc[pred_feats].sort_index()

    
    
    if int_struct:    
        print("=========== Internal structure in cluster==============")
        for cl_id in range(n_clust):        
            print("cluster ====>",cl_id)
            cres = {}

            cov = bgm.covariances_[cl_id]        
            import scipy
            (eigvals, eigvecs) = scipy.linalg.eigh(bgm.covariances_[cl_id])
            print("np.sqrt(eigvals)")
            print(np.sqrt(eigvals))

            cl_elong_a = np.sqrt(eigvals[-1]/eigvals[-2])
            cl_elong_v = eigvecs[-1]
            print("cl_elong_a, cl_elong_v.round(2)")
            print(cl_elong_a, cl_elong_v.round(2))  


            cres["cl_elong_a"] = cl_elong_a
            cres["cl_elong_v"] = cl_elong_v
            cres["cl_elong_var"] = eigvals[-1]
            cres["cl_mean"] = bgm.means_[cl_id]

            print("cluster data sample, mean shift")

            cl_ica_data = my_ica_data[bgm.predict(my_ica_data)==cl_id] - bgm.means_[cl_id]
            cl_ica_data=cl_ica_data[~cl_ica_data.index.duplicated(keep='first')]
            print(cl_ica_data.shape, cl_ica_data.head(3))

            print("cluster data elongation along main axis")        
            # global token_elong
            token_elong = pd.Series(np.inner(cl_ica_data,cl_elong_v),name='x',index=cl_ica_data.index)
            token_elong.head(3)

            # Compute each feature's effect on the projected axis


            cl_feat_elong_ip = pd.Series(np.inner(feat_v,cl_elong_v),
                                         index=feat_v.index,
                                         name='effect').loc[clust_featdx_df.loc[cl_id].index].sort_values(ascending=False)

            cl_feat_elong_df = pd.concat([cl_feat_elong_ip,clust_featdx_df.loc[cl_id]],axis=1)
            cl_feat_elong_df['eff2']=cl_feat_elong_df.effect**2

            cl_feat_elong_top = cl_feat_elong_df.sort_values('eff2',ascending=False).head(10)
            print(cl_feat_elong_top)

            print("Compute the feature-associated token x1 positions (ftok) on the projected axis")
            
            # global ftok

            ftok = feat_tokens.loc[cl_feat_elong_top.index]
            ftok = ftok.merge(token_elong,left_on='token',right_index=True).sort_index()
            ftok = ftok.reset_index().drop_duplicates().set_index(['arcpath','lemma'])
            print("ftok")
            print(ftok)


            print("Evaluate the feature-associated token positions on the projected axis")
            print("Compare to the group of tokens not associated with the feature (X0)")

            res = []
            feat_index = ftok.index.unique()
            
            print('feat_index',feat_index.shape)
            print('token_elong',token_elong.shape)
            print('ftok',ftok.shape)            
            
            
            for feat_i in feat_index:
                #print ('feat_i',feat_i)
                x1 = ftok.loc[[feat_i]].x
                tokens_in = ftok.loc[[feat_i]].token
                
                #print('tokens_in',tokens_in)
                #print('x1',x1)                
                
                
                x0 = token_elong[~token_elong.index.isin(tokens_in)]

                print('x0',x0)                

                n1 = len(x1)                
                n0 = len(x0)
                dx = np.round(x1.mean()-x0.mean(),3)

                stat=st.ttest_ind(x1, x0)

                res.append({'test_pvalue':np.round(stat.pvalue,5),
                            'test_dX':dx,'test_Nx':n1,'test_N':n1+n0,'test_st':np.round(stat.statistic,2)})
                    
            if len(res):


                feat_tests = pd.DataFrame(res,index = feat_index)
                print("Spearman results")
                print("feat_tests")
                print(feat_tests)

                book_cl_elong = pd.concat([cl_feat_elong_top,feat_tests],axis=1)        
                book_cl_elong = book_cl_elong[book_cl_elong.test_dX * book_cl_elong.effect > 0]
                print("book_cl_elong")
                print(book_cl_elong)

                cres["cl_ica_data"] = cl_ica_data
                cres["cl_feat_elong_top"] = cl_feat_elong_top
                cres["book_cl_elong"] = book_cl_elong
                cres["token_elong"] = token_elong


                result['C%d'%cl_id] = cres
    
    print("===========  end ==============")
    
    
    return result




In [318]:
cresTax = analyze_category('taxongroup',True)


checking for data/q_cluster/BGM_full_covar_taxongroup.pkl
BGM loaded
bgm.predict(my_ica_data)).value_counts()
9    2472
2    2429
6    1400
0     842
8     552
4     282
1     233
7     134
3      12
5       5
Name: count, dtype: int64
clust_feats (19040, 2)
                 cluster  freq
arcpath   lemma               
case>     of           6   818
det>      the          2   613
          the          9   441
nmod<det> the          6   379
case>     of           2   346
clust_f_df pivot table (10866, 10)
clust_p_df is the probability distribution cluster-wise
                              0    1         2    3    4    5    6    7    8  \
arcpath              lemma                                                     
acl:relcl<case>      of     0.0  0.0  0.000096  0.0  0.0  0.0  0.0  0.0  0.0   
acl:relcl<det>       the    0.0  0.0  0.000192  0.0  0.0  0.0  0.0  0.0  0.0   
acl:relcl<obl<cop>   be     0.0  0.0  0.000096  0.0  0.0  0.0  0.0  0.0  0.0   
acl:relcl<obl<nsubj> it     0.0  

x0 token_id
Arc-b4f2_60_Sterna-17         0.273782
Cir-aed8_182_animal-20       -0.651327
Fri-5711_17_Fritillaria-1    -0.375972
Atl-12cc_25_Istiophorus-11    0.063683
Hem-5b5a_21_arthropod-29     -0.094266
                                ...   
Pie-14e4_111_Pieris-1        -0.362709
Rig-c8bd_97_Eubalaena-7       0.158882
Pte-62aa_313_reptile-27      -0.030494
The-f5b4_16_dinosaur-18       0.081733
Wil-3cbf_107_Accipiter-42     0.068505
Name: x, Length: 826, dtype: float64
x0 token_id
Arc-b4f2_60_Sterna-17         0.273782
Cir-aed8_182_animal-20       -0.651327
Fri-5711_17_Fritillaria-1    -0.375972
Atl-12cc_25_Istiophorus-11    0.063683
Hem-5b5a_21_arthropod-29     -0.094266
                                ...   
Pie-14e4_111_Pieris-1        -0.362709
Rig-c8bd_97_Eubalaena-7       0.158882
Pte-62aa_313_reptile-27      -0.030494
The-f5b4_16_dinosaur-18       0.081733
Wil-3cbf_107_Accipiter-42     0.068505
Name: x, Length: 826, dtype: float64
x0 token_id
Arc-b4f2_60_Sterna-17         0.

book_cl_elong
                        effect        Dx   N      eff2  test_pvalue  test_dX  \
arcpath    lemma                                                               
cc>        and       -0.751177  0.002718  13  0.564267      0.03505   -0.631   
nmod>case> of         0.729722  0.004417   8  0.532494      0.20712    0.477   
nmod>      Hewitson   0.725210  0.001158   1  0.525929      0.25272    1.202   
nsubj<     order      0.717932  0.001071   1  0.515427      0.40541    0.875   
nmod>case> in         0.702984  0.005788   9  0.494187      0.48678    0.249   
cc>        as        -0.698172  0.002902   3  0.487444      0.44919   -0.462   
nmod>      Boisduval  0.672279  0.001131   1  0.451959      0.30727    1.073   
nmod>case> with       0.668601  0.002075   4  0.447027      0.32006    0.527   
           as         0.662159  0.000889   1  0.438454      0.30989    1.068   

                      test_Nx  test_N  test_st  
arcpath    lemma                                
cc>    

ftok
                                           token         x
arcpath         lemma                                     
amod>           cuttlefish   Sep-9969_11_Sepia-8 -0.579888
compound>       Sepia        Sep-9969_11_Sepia-8 -0.579888
                Sepia       Sep-9969_11_Sepia-94 -0.244920
                cuttlefish  Sep-9969_11_Sepia-30 -0.055540
flat<compound>  Sepia        Sep-9969_11_Sepia-8 -0.579888
...                                          ...       ...
flat<list>amod> giant       Sep-9969_11_Sepia-44 -0.051512
                giant       Sep-9969_11_Sepia-46 -0.051512
                giant       Sep-9969_11_Sepia-48 -0.051512
                giant       Sep-9969_11_Sepia-50 -0.051512
                giant       Sep-9969_11_Sepia-52 -0.051512

[87 rows x 2 columns]
Evaluate the feature-associated token positions on the projected axis
Compare to the group of tokens not associated with the feature (X0)
feat_index (10,)
token_elong (12,)
ftok (87, 2)
x0 token_id
Sep-996

/home/seppo/.local/lib/python3.10/site-packages/scipy/_lib/deprecation.py:234: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  return f(*args, **kwargs)
/home/seppo/.local/lib/python3.10/site-packages/scipy/_lib/deprecation.py:234: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  return f(*args, **kwargs)
/home/seppo/.local/lib/python3.10/site-packages/scipy/_lib/deprecation.py:234: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  return f(*args, **kwargs)
/home/seppo/.local/lib/python3.10/site-packages/scipy/_lib/deprecation.py:234: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  return f(*args, **kwar

...

/home/seppo/.local/lib/python3.10/site-packages/scipy/_lib/deprecation.py:234: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  return f(*args, **kwargs)
/home/seppo/.local/lib/python3.10/site-packages/scipy/_lib/deprecation.py:234: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  return f(*args, **kwargs)
/home/seppo/.local/lib/python3.10/site-packages/scipy/_lib/deprecation.py:234: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  return f(*args, **kwargs)
/home/seppo/.local/lib/python3.10/site-packages/scipy/_lib/deprecation.py:234: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  return f(*args, **kwar

(1332, 8)                                    0         1         2         3         4  \
token_id                                                                       
Hor-ab9c_184_bat-13        -0.438082 -0.355401  0.229085 -1.898982  1.375143   
The-1978_215_Saurischia-16  0.246799 -0.374635 -0.626796 -0.283815  0.974614   
Com-f222_479_birds-11       0.784924 -0.106542 -0.386290  1.482840  0.764857   

                                   5         6         7  
token_id                                                  
Hor-ab9c_184_bat-13        -0.538853 -0.121674  0.336166  
The-1978_215_Saurischia-16  0.048036 -0.296486  0.502807  
Com-f222_479_birds-11      -0.594153  0.128833  0.224974  
cluster data elongation along main axis
                                    effect        Dx   N      eff2
arcpath              lemma                                        
acl:relcl>           include     -0.771728  0.001694  10  0.595564
                     contain     -0.753686  0.000493 

ftok
                                             token         x
arcpath    lemma                                            
amod>      black              Com-3b7c_19_birds-36 -4.716474
           black               Eur-35e8_60_gull-19 -3.349200
           gray             Sci-501e_7_squirrel-17 -0.368754
nmod>case> from             Cir-aed8_63_animals-42 -2.967543
           from                 Arc-b4f2_43_bird-8 -3.705202
           from            Lea-dca1_2_dinosaurs-17 -1.707399
           in                   Roe-b5ac_55_deer-8 -3.978991
           in                 Euo-1402_2_spider-13 -3.421679
           in               Rho-d4a0_238_plants-15 -4.881607
           in                Bur-9073_53_plants-17 -2.615279
           in                   Vio-0709_9_plant-6 -3.684067
           in                  Mar-9b95_2_mammal-8 -4.075736
           in                 Hem-78f3_7_insects-2 -5.043062
           in                   Roo-1bd1_22_bird-7 -3.286340
           in      

                        effect        Dx   N      eff2
arcpath    lemma                                      
case>      from       0.512475 -0.000266   1  0.262630
acl:relcl> fly       -0.507750  0.000312   1  0.257810
obl<       higher     0.504366  0.000189   1  0.254385
conj<conj> waterfowl -0.500098  0.000708   2  0.250098
case>      of         0.497451 -0.001521   3  0.247457
conj<conj> marmot    -0.496653  0.001062   3  0.246664
           hedgehog  -0.487617  0.000708   2  0.237771
           squirrel  -0.486893  0.006020  17  0.237065
           seabird   -0.485202  0.000708   2  0.235421
case>      in         0.482628 -0.000417   1  0.232930
Compute the feature-associated token x1 positions (ftok) on the projected axis
ftok
                                            token         x
arcpath    lemma                                           
acl:relcl> fly              Com-f222_150_birds-26  0.481457
case>      from              Hor-ab9c_205_bats-17  1.041855
           in   

ftok
                                   token         x
arcpath lemma                                     
amod>   most   Art-1308_168_arthropods-2  0.508565
        most       Bov-d455_162_bovids-2  0.793022
        most      Col-6d94_171_plants-29 -0.619072
        most         Dra-242f_8_plants-4  1.234355
        most        Eur-35e8_138_bird-12  1.027774
...                                  ...       ...
det>    some         Too-567c_25_birds-2 -0.684618
        some       Too-567c_343_birds-33 -0.566455
        some       Too-567c_48_animals-2 -1.721029
        some      Ven-071d_69_lizards-23 -1.025333
        those     Swa-27b4_32_animals-36 -0.862943

[130 rows x 2 columns]
Evaluate the feature-associated token positions on the projected axis
Compare to the group of tokens not associated with the feature (X0)
feat_index (10,)
token_elong (2384,)
ftok (130, 2)
x0 token_id
Sci-501e_7_squirrel-5        0.389883
Ven-77c9_97_arthropods-16    1.290315
Ham-9078_102_sharks-13      -0.

In [152]:
token_elong

token_id
Sci-501e_7_squirrel-5        0.389883
Ven-77c9_97_arthropods-16    1.290315
Ham-9078_102_sharks-13      -0.757125
Ani-161e_84_animals-4        0.356346
Cro-0859_276_crocodile-61    0.667533
                               ...   
Cet-5f63_400_animals-17      1.231640
Jel-28f4_51_jellyfish-11     0.488976
Gree5e5_4_beetles-3         -0.000649
Art-1308_168_arthropods-2    0.508565
Rep-aa64_32_animals-26       0.039188
Name: x, Length: 2384, dtype: float64

In [80]:
res = analyze_category('adaptation')

checking for data/q_cluster/BGM_full_covar_adaptation.pkl
BGM loaded
bgm.predict(my_ica_data)).value_counts()
1    754
6    631
2    426
3    378
4    321
8    307
9    306
5    269
0    130
7     56
Name: count, dtype: int64
clust_feats (7134, 2)
                  cluster  freq
arcpath    lemma               
cop>       be           1   159
           be           9   140
case>      of           8   131
conj>cc>   and          9   100
amod<case> of           6    87
clust_f_df pivot table (3355, 10)
clust_p_df is the probability distribution cluster-wise
                               0         1         2    3        4    5    6  \
arcpath              lemma                                                     
acl:relcl<           theory  0.0  0.000583  0.000000  0.0  0.00000  0.0  0.0   
acl:relcl<amod>      other   0.0  0.000000  0.000000  0.0  0.00132  0.0  0.0   
acl:relcl<case>      to      0.0  0.000000  0.000000  0.0  0.00066  0.0  0.0   
acl:relcl<det>       the     0.0  0.00

In [82]:
res['clust_featdx_df_short']

Dx    N
cluster arcpath          lemma                   
0       cc>              and        0.095173   45
        conj<conj>cc>    and        0.083995   31
        conj<cop>        be         0.067975   28
        conj<det>        the        0.037252   14
                         a          0.026249   10
1       cop>             be         0.042825  159
        conj>cc>         and        0.017637   64
        obj<             have       0.014957   33
        nsubj<cop>       be         0.013444   51
        compound<det>    the        0.013096   33
2       nmod>det>        the        0.021774   61
        det>             the        0.020928   75
        nmod>case>fixed> as         0.019305   42
        nmod>case>       such       0.019305   42
                         in         0.013629   35
3       amod<cc>         and        0.101903   35
        amod<            behavior   0.070457   25
                         behaviour  0.047395   17
        compound<case>   of         0.037640   14
        amod<            structure  0.036440   15
4       obl<aux:pass>    be         0.053872   42
        obl<mark>        to         0.042485   30
        case>            from       0.039231   33
                         as         0.030477   27
                         by         0.026708   22
5       nmod>case>       of         0.051056   72
        nsubj<cop>       be         0.043491   43
        det>             the        0.033776   47
        nsubj<conj>cc>   and        0.031434   36
        nmod>det>        the        0.022905   31
6       amod<case>       of         0.077156   87
        amod<det>        the        0.060507   74
                         a          0.049745   64
        amod<case>       in         0.039302   45
        amod<cop>        be         0.033319   46
7       nsubj<           include    0.254891   33
        nmod>case>       of         0.093501   18
        nsubj<obj>conj>  fox        0.066827    8
                         wolf       0.050606    6
                         bird       0.049961    6
8       case>            of         0.196415  131
        nmod<det>        the        0.125495   80
        amod>            human      0.081236   61
        case>            in         0.063661   47
        nmod<det>        a          0.057369   38
9       conj>cc>         and        0.065120  100
        cop>             be         0.058856  140
        nsubj>det>       the        0.023393   38
        conj>cc>         but        0.016707   21
        conj>nmod>case>  of         0.012190   14

True

In [83]:
cresTop = analyze_category('topic',True)


checking for data/q_cluster/BGM_full_covar_topic.pkl
BGM loaded
bgm.predict(my_ica_data)).value_counts()
4    8896
0    6276
5    5179
8    5004
1    4837
9    3572
6    2254
7    1892
3    1364
2     259
Name: count, dtype: int64
clust_feats (80022, 2)
                  cluster  freq
arcpath    lemma               
case>      of           5  2803
det>       the          4  2236
cc>        and          9  1723
nsubj<cop> be           4  1693
conj>cc>   and          0  1583
clust_f_df pivot table (39715, 10)
clust_p_df is the probability distribution cluster-wise
                                0         1    2    3         4    5  \
arcpath         lemma                                                  
acl:relcl<      animal   0.000041  0.000022  0.0  0.0  0.000000  0.0   
acl:relcl<case> in       0.000020  0.000000  0.0  0.0  0.000000  0.0   
                include  0.000041  0.000000  0.0  0.0  0.000000  0.0   
                of       0.000061  0.000044  0.0  0.0  0.000018  0.0   

ftok
                                                token         x
arcpath    lemma                                               
conj<conj> Gallai                 Gir-cc25_27_Rook-83 -1.532002
           capuching          Ceb-b443_19_capuchin-18 -1.070245
           eagle               Nor-4d51_285_eagles-16 -0.514836
           martin              Coma03b_19_swallows-18 -1.121982
conj>      Cricetidae          New-caad_18_Rodentia-5  0.755989
           Cricetidae          Shr-2a0f_11_Muridae-14  1.026054
           Ornithischia     Pte-62aa_25_Saurischia-33  1.014217
           Ornithischia      Sau-6bc5_51_Theropoda-20  0.693645
           Reptilia           Cro-9102_581_Amphibia-5  0.314303
           Saurischia    Orn-f740_113_Ornithischia-35  1.327374
           Stegosauria     Thy-db39_14_Ankylosauria-8  0.449961
           Stegosauria    Thy-db39_56_Ankylosauria-28  1.071223
           Stegosauria      Thy-db39_6_Ankylosauria-9  0.736581
           nigricollis            H

/home/seppo/.local/lib/python3.10/site-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


                           effect        Dx  N      eff2
arcpath         lemma                                   
nsubj<obj>conj> bird    -1.275181  0.000086  2  1.626086
                buffalo -1.248956  0.000037  1  1.559891
                snake   -1.129129  0.000086  2  1.274932
                turtle  -1.112352  0.000003  2  1.237327
                shark   -1.092524  0.000050  1  1.193609
                colour  -1.079654  0.000037  1  1.165653
                species -1.072816  0.000028  3  1.150934
                mouse   -1.057641  0.000050  1  1.118604
                frog    -1.033550  0.000028  3  1.068226
                rodent  -1.025442  0.000037  1  1.051531
Compute the feature-associated token x1 positions (ftok) on the projected axis
ftok
                                              token         x
arcpath         lemma                                        
nsubj<obj>conj> bird     Boa-7d46_71_constrictors-3 -4.525416
                bird         Kom-6ea1_112_drag

ftok
                                             token         x
arcpath            lemma                                    
amod>              greater       Gre-c26f_74_bat-3 -0.070319
                   high     Hig81e8_2_fritillary-3  0.039973
                   high     Vio6842_8_fritillary-3  0.039973
amod>advmod>       long     Lon-6fc7_10_seahorse-4  0.039973
                   long     Lon-6fc7_17_seahorse-4 -0.105636
...                                            ...       ...
compound>compound> New          New-caad_12_rats-3 -0.019702
conj>              mouse        New-caad_10_rats-3 -0.165546
                   mouse        New-caad_12_rats-3 -0.019702
nmod>case>         of             Law5343_5_Bird-1  0.039973
                   of         New-d8c1_127_Birds-1  0.039973

[63 rows x 2 columns]
Evaluate the feature-associated token positions on the projected axis
Compare to the group of tokens not associated with the feature (X0)
Spearman results
feat_tests
             

...

ftok
                                    token         x
arcpath lemma                                      
conj>   animal     Ani-2170_118_humans-20 -0.166233
        animal       Ara-8421_54_plant-22 -3.080116
        animal    Arm-33e3_135_insects-25 -1.983136
        animal      Asc-4810_139_plants-4 -0.833943
        animal      Aus-f9df_11_plants-23  0.789910
...                                   ...       ...
        rabbit   Har-8a11_65_squirrels-13 -0.915657
        rabbit       Her-8d89_180_deer-15 -1.276406
        rabbit    Man-d9fd_119_rodents-16 -2.168446
        variety     Ech-581f_68_rodents-5 -1.647732
        variety      Flo-eb50_8_plants-40 -0.596479

[71 rows x 2 columns]
Evaluate the feature-associated token positions on the projected axis
Compare to the group of tokens not associated with the feature (X0)
Spearman results
feat_tests
                     test_pvalue  test_dX  test_Nx  test_N  test_st
arcpath lemma                                                 

(4977, 8)                                0         1         2         3         4  \
token_id                                                                   
Afr-655f_81_elephant-2 -0.268693  0.213770 -0.599649 -0.286494 -0.673381   
The067f_13_camels-3    -0.380279  0.302888 -0.037307  0.027421 -0.036201   
Hun-23f5_11_spiders-6   0.664105  0.060592  0.619119 -0.080039 -0.191112   

                               5         6         7  
token_id                                              
Afr-655f_81_elephant-2  0.378184 -0.152703  0.051157  
The067f_13_camels-3     0.060925 -0.032873  0.165299  
Hun-23f5_11_spiders-6   0.141119 -0.007770 -0.355236  
cluster data elongation along main axis
                                    effect        Dx  N      eff2
arcpath                 lemma                                    
case>                   by       -0.510827 -0.000188  1  0.260945
compound<amod>compound> New      -0.505260  0.000420  3  0.255288
case>                   '     

/home/seppo/.local/lib/python3.10/site-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


Spearman results
feat_tests
                                  test_pvalue  test_dX  test_Nx  test_N  \
arcpath                 lemma                                             
amod<                   ancestor      0.05373    0.602        1    4977   
appos<nsubj<            know          0.25770    0.353        1    4977   
case>                   '             0.10964   -0.499        1    4977   
                        by            0.31667   -0.312        1    4977   
                        from          0.00637   -0.602        2    4977   
                        of            0.18844   -0.184        5    4977   
compound<amod>compound> New           0.00259   -0.543        3    4977   
compound<obl>case>      in            0.00090    0.598        3    4977   
compound>det>           the           0.07786    0.550        1    4977   
obl<                    faster        0.19561   -0.404        1    4977   

                                  test_st  
arcpath                 lem

/home/seppo/.local/lib/python3.10/site-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


In [84]:
cresTop.keys()

dict_keys(['clust_featdx_df_short', 'clust_featdx_df', 'cos_pvalue', 'score_pvalue', 'C0', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9'])

In [85]:
cresTop['score_pvalue']

cos_stat  cos_pvalue  score_stat  score_pvalue
arcpath            lemma                                                  
det>               a        0.648485    0.042540    0.869175      0.001091
case>              from     0.842424    0.002220    0.869175      0.001091
                   as       0.214740    0.551325    0.860915      0.001379
                   to       0.187592    0.603782    0.860915      0.001379
                   's       0.382980    0.274673    0.859276      0.001443
...                              ...         ...         ...           ...
conj<det>          the      0.006079    0.986703   -0.043905      0.904144
conj<obl<aux:pass> be      -0.398803    0.253627   -0.037982      0.917035
conj<conj>cc>      and      0.325178    0.359238   -0.037383      0.918338
conj<case>fixed>   as      -0.135767    0.708424   -0.013341      0.970822
amod>              african -0.248485    0.488776    0.006213      0.986409

[100 rows x 4 columns]

In [360]:
cresAda = analyze_category('adaptation',True)
cresLoc = analyze_category('location',True)
cresTax = analyze_category('taxongroup',True)
cresTop = analyze_category('topic',True)


checking for data/q_cluster/BGM_full_covar_adaptation.pkl
BGM loaded
bgm.predict(my_ica_data)).value_counts()
1    754
6    631
2    426
3    378
4    321
8    307
9    306
5    269
0    130
7     56
Name: count, dtype: int64
clust_feats (7134, 2)
                  cluster  freq
arcpath    lemma               
cop>       be           1   159
           be           9   140
case>      of           8   131
conj>cc>   and          9   100
amod<case> of           6    87
clust_f_df pivot table (3355, 10)
clust_p_df is the probability distribution cluster-wise
                               0         1         2    3        4    5    6  \
arcpath              lemma                                                     
acl:relcl<           theory  0.0  0.000583  0.000000  0.0  0.00000  0.0  0.0   
acl:relcl<amod>      other   0.0  0.000000  0.000000  0.0  0.00132  0.0  0.0   
acl:relcl<case>      to      0.0  0.000000  0.000000  0.0  0.00066  0.0  0.0   
acl:relcl<det>       the     0.0  0.00

ftok
                                         token         x
arcpath    lemma                                        
amod>      ambush       Amb-b5dc_7_predators-3  0.567340
           ambush      Pyt-d6ca_30_predators-8  0.690301
           pure      Smi-7190_144_scavenger-10 -0.009813
           pure       Tyr-9f97_14_scavenger-13 -1.454685
           pure      Tyr-9f97_537_scavenger-17 -1.065960
nmod>case> for        Aur-441a_25_predators-13  0.280296
           from        Lan-4994_75_language-20  1.064271
           in        Aur-c4ea_65_dimorphism-39  0.920155
           in        Cro-0859_136_predators-17  1.002693
           in         Kāk-e004_136_predators-2 -0.314572
           in          Mam-3742_19_language-36  1.211793
           in        Nil-600c_341_predators-25  1.722447
           include   Bee-ace3_329_predators-37  1.815256
           include    Com-0bc9_90_predators-17  0.560875
           like       Cap-5548_62_predators-34  0.770489
           like       The-

ftok
                                            token         x
arcpath    lemma                                           
cc>        rather      Cam-4bd8_115_camouflage-34 -1.081093
           rather       Lon-b0af_519_predators-18 -0.604779
conj>      arboreal       Aye-787b_52_nocturnal-7  0.167632
           arboreal        Uro-49a1_7_nocturnal-7  0.325588
           find            Uro-49a1_3_nocturnal-3  0.560101
           group            Col-6d94_129_social-9  0.872088
           have          Thy-db39_5_herbivorous-4  0.556646
           have             Vip0c87_4_predators-5  0.542191
           omnivorous  Bov-d455_138_herbivorous-7  0.912342
           omnivorous  Mur-6918_12_herbivorous-14  0.569031
           omnivorous  Sau-6bc5_60_herbivorous-20  0.490583
           omnivorous  Sau-cca0_13_herbivorous-35  1.156094
           oviparous     Viv-ac84_84_viviparous-9  0.457231
           oviparous     Viv-ac84_85_viviparous-3  0.645998
           scavenger      Neo-e8a9_

ftok
                                                  token         x
arcpath              lemma                                       
nmod>nmod>           Africa     Bar-b361_99_Migration-1 -1.056930
                     Africa    Smi-7190_210_predator-11  0.922894
                     Africa     Whi-100a_42_migration-9 -0.202672
nsubj:pass<obl>case> with      Asi-f65e_70_dimorphism-2 -5.536542
                     with       Bil-e183_43_Predators-1 -1.391066
                     with         Lan-4994_4_language-2 -2.434696
                     with      Lon-b0af_249_migration-2 -2.812940
                     with           Pat-9307_138_care-3 -2.663568
nsubj<acl>advmod>    not         Lan-4994_95_language-6  0.430549
nsubj<obj>conj>      song       Ani-4d8b_157_language-2  2.891154
obj<advcl<advmod>    also     Eur-bc75_418_predators-12 -0.302634
obl<                 compete  Eur-bc75_370_predators-17  1.650754
                     compete   Gia-ae2f_21_predators-17  0.819770
     

ftok
                                                  token         x
arcpath             lemma                                        
advmod>             however   Eas-6961_45_territorial-6 -0.468134
amod<amod>advmod>   highly        Pan-3679_96_social-21  0.059674
case>               unlike       Amb-b5dc_3_predators-3 -0.364435
compound<nmod<case> in         Rat-4b67_135_predator-17 -0.073609
                    of         Lan-4994_112_language-14 -0.273776
                    of          Pas-a544_2_flowering-21 -0.385331
                    of        Wal-8f46_154_predators-16 -0.179538
compound<nmod<cc>   and         Orc-02f3_16_cultures-17 -0.269129
                    and       Swa-27b4_296_migration-21 -0.227935
compound>           Reindeer     Ani6f2b_67_migration-2  0.328596
                    Reindeer      Rei5e90_9_migration-2  0.328596
det>                a              Wes30d4_16_display-5 -0.157579
                    no         Ruf-ac2f_10_dimorphism-6 -0.390874
     

...

                      effect        Dx   N      eff2
arcpath    lemma                                    
nmod>case> in      -0.804104 -0.000571  10  0.646584
           over    -0.795272  0.000032   1  0.632457
           to      -0.791271 -0.000080   3  0.626110
           of      -0.768651 -0.000595  16  0.590824
           from    -0.722810 -0.000135   3  0.522454
           on      -0.698936 -0.000101   3  0.488511
           include -0.688765 -0.000100   3  0.474397
           with    -0.671022 -0.000149   2  0.450271
           for     -0.668740 -0.000069   1  0.447214
           such    -0.652954  0.000141  26  0.426349
Compute the feature-associated token x1 positions (ftok) on the projected axis
ftok
                                       token         x
arcpath    lemma                                      
nmod>case> for             Aphe0a9_6_cows-18 -0.703107
           from       Cir-aed8_249_Circus-47 -1.354382
           from       Cir-aed8_249_Circus-54 -1.187210
     

(4977, 8)                                0         1         2         3         4  \
token_id                                                                   
Afr-655f_81_elephant-2 -0.268693  0.213770 -0.599649 -0.286494 -0.673381   
The067f_13_camels-3    -0.380279  0.302888 -0.037307  0.027421 -0.036201   
Hun-23f5_11_spiders-6   0.664105  0.060592  0.619119 -0.080039 -0.191112   

                               5         6         7  
token_id                                              
Afr-655f_81_elephant-2  0.378184 -0.152703  0.051157  
The067f_13_camels-3     0.060925 -0.032873  0.165299  
Hun-23f5_11_spiders-6   0.141119 -0.007770 -0.355236  
cluster data elongation along main axis
                                    effect        Dx  N      eff2
arcpath                 lemma                                    
case>                   by       -0.510827 -0.000188  1  0.260945
compound<amod>compound> New      -0.505260  0.000420  3  0.255288
case>                   '     

/home/seppo/.local/lib/python3.10/site-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)
/home/seppo/.local/lib/python3.10/site-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


x0 token_id
Afr-655f_81_elephant-2    0.236092
The067f_13_camels-3      -0.153989
Hun-23f5_11_spiders-6    -0.031079
Auk-a4a3_24_divers-22     0.307631
Pin-32d8_16_pine-2       -0.674474
                            ...   
Wes-8d71_327_bee-3        0.171030
Col-25ad_103_Columba-2   -0.309843
Tyrfbca_31_rex-2         -0.417014
Cul-6926_25_human-15      0.164772
Ala-9ac8_15_Alauda-1     -0.399550
Name: x, Length: 4974, dtype: float64
x0 token_id
Afr-655f_81_elephant-2    0.236092
The067f_13_camels-3      -0.153989
Hun-23f5_11_spiders-6    -0.031079
Auk-a4a3_24_divers-22     0.307631
Pin-32d8_16_pine-2       -0.674474
                            ...   
Wes-8d71_327_bee-3        0.171030
Col-25ad_103_Columba-2   -0.309843
Tyrfbca_31_rex-2         -0.417014
Cul-6926_25_human-15      0.164772
Ala-9ac8_15_Alauda-1     -0.399550
Name: x, Length: 4976, dtype: float64
x0 token_id
Afr-655f_81_elephant-2    0.236092
The067f_13_camels-3      -0.153989
Hun-23f5_11_spiders-6    -0.031079
Auk-a4a3_24_d

/home/seppo/.local/lib/python3.10/site-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)
/home/seppo/.local/lib/python3.10/site-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


In [361]:
cresTax['C1']

{'cl_elong_a': 2.2333079518709735,
 'cl_elong_v': array([ 0.55751149,  0.06348049,  0.78398189, -0.21571017,  0.04214997,
        -0.09931228, -0.10830427,  0.02496623]),
 'cl_elong_var': 7.8649476230195035,
 'cl_mean': array([ 0.88280341,  0.55957607, -0.31770235,  0.4087019 , -0.26105165,
         0.39534534, -0.19086827,  0.47535111]),
 'cl_ica_data':                                     0         1         2         3         4  \
 token_id                                                                        
 Tru-f072_21_animals-7        0.968425 -3.946513 -0.182949  1.656552  1.416039   
 Din-756f_27_Dinosauria-7     0.259754 -2.546674  0.521633  0.218453  0.434535   
 Sca-9161_30_Animals-1        0.173958 -2.289524  0.608234  0.203571  0.506868   
 Hor-ab9c_225_bat-8          -0.556522 -1.916160  0.564865 -0.502813  0.630258   
 Dun-cad0_87_beetles-3        0.858886 -3.028922  0.490710 -1.067172 -0.262849   
 ...                               ...       ...       ...       ...  

In [367]:
def cleanlatex(df, screen=True):
    s = df.reset_index().to_latex(index=False).replace('<','< ').replace('>','> ').\
          replace('0000 & ',' & ').replace('000 & ',' & ').replace('0000 \\',' \\').replace('000 \\',' \\')
    if screen:
        print(s)
        return
    return s


In [362]:
res = cresAda # take  data

In [369]:
res = cresLoc # take  data

In [375]:
res = cresTax # take  data

In [381]:
res = cresTop # take  data

In [382]:
n_df = pd.DataFrame(
    {'Nfeat':res['clust_featdx_df'].groupby(['arcpath','lemma']).N.sum(),
     'Nclust':res['clust_featdx_df'].reset_index().groupby(['arcpath','lemma']).cluster.count()})

n_df = n_df[(n_df.Nclust>2)|(n_df.Nfeat>=10)]


dx_eval = res['clust_featdx_df'].merge(n_df,left_index=True, right_index=True)
sc_eval = res['score_pvalue'].iloc[:,2:4].merge(n_df,left_index=True, right_index=True)
cd_eval = res['cos_pvalue'].iloc[:,0:2].merge(n_df,left_index=True, right_index=True)

dx_eval = dx_eval.sort_values(by=['Dx','Nclust'],ascending=False)
dx_eval = dx_eval.groupby(['cluster']).head(3)
dx_eval = dx_eval.groupby(['lemma']).head(1).drop(columns=['N'])
sc_eval = sc_eval.groupby(['lemma']).head(1).rename(columns={'score_pvalue':'p-value','score_stat':'log-likelihood Rho'})
cd_eval = cd_eval.groupby(['lemma']).head(1).rename(columns={'cos_pvalue':'p-value','cos_stat':'cosine Rho'})

dx_eval = dx_eval.sort_values(by='Dx',ascending=False).head(10)
sc_eval = sc_eval.sort_values(by='p-value').head(10)
cd_eval = cd_eval.sort_values(by='p-value').head(10)

In [383]:
dx_eval


Dx  Nfeat  Nclust
arcpath          lemma       cluster                         
case>            of          5        0.158360   4186       8
cc>              and         9        0.123504   2902       8
appos>           description 3        0.097598    144       3
nmod<det>        the         5        0.089275   2094       6
appos>           reference   3        0.072228     83       3
                 Taxonomy    3        0.066884     92       4
nsubj<cop>       be          4        0.053416   2413       9
nmod<det>        a           5        0.032987    951       8
conj<case>fixed> as          9        0.031931    619       5
conj<obj<        include     7        0.012131    581       7

In [384]:
sc_eval


log-likelihood Rho   p-value  Nfeat  Nclust
arcpath          lemma                                             
det>             a                0.869175  0.001091   1446       9
case>            from             0.869175  0.001091    366       9
                 as               0.860915  0.001379    508       7
                 to               0.860915  0.001379    780       6
                 's               0.859276  0.001443    427       8
obl<nsubj>det>   the              0.857382  0.001518    325       5
case>            of               0.827916  0.003100   4186       8
                 with             0.822581  0.003479    578       6
acl:relcl>nsubj> that             0.812897  0.004249    280       9
case>            for              0.769126  0.009307    488       9

In [385]:
cd_eval


cosine Rho   p-value  Nfeat  Nclust
arcpath         lemma                                       
case>           like       0.863226  0.001294    327       8
case>fixed>     as         0.852825  0.001712    703       7
case>           with       0.844162  0.002128    578       6
nmod>case>      in         0.842424  0.002220    681       9
case>           from       0.842424  0.002220    366       9
                for        0.793939  0.006100    488       9
det>            some       0.790277  0.006514    277       8
case>           of         0.765961  0.009787   4186       8
conj<conj>amod> other      0.730554  0.016409    245       5
conj<case>      include    0.689617  0.027350    302       4

In [386]:
cleanlatex(dx_eval)
cleanlatex(sc_eval)
cleanlatex(cd_eval)

\begin{tabular}{llrrrr}
\toprule
arcpath & lemma & cluster & Dx & Nfeat & Nclust \\
\midrule
case>  & of & 5 & 0.158360 & 4186 & 8 \\
cc>  & and & 9 & 0.123504 & 2902 & 8 \\
appos>  & description & 3 & 0.097598 & 144 & 3 \\
nmod< det>  & the & 5 & 0.089275 & 2094 & 6 \\
appos>  & reference & 3 & 0.072228 & 83 & 3 \\
appos>  & Taxonomy & 3 & 0.066884 & 92 & 4 \\
nsubj< cop>  & be & 4 & 0.053416 & 2413 & 9 \\
nmod< det>  & a & 5 & 0.032987 & 951 & 8 \\
conj< case> fixed>  & as & 9 & 0.031931 & 619 & 5 \\
conj< obj<  & include & 7 & 0.012131 & 581 & 7 \\
\bottomrule
\end{tabular}

\begin{tabular}{llrrrr}
\toprule
arcpath & lemma & log-likelihood Rho & p-value & Nfeat & Nclust \\
\midrule
det>  & a & 0.869175 & 0.001091 & 1446 & 9 \\
case>  & from & 0.869175 & 0.001091 & 366 & 9 \\
case>  & as & 0.860915 & 0.001379 & 508 & 7 \\
case>  & to & 0.860915 & 0.001379 & 780 & 6 \\
case>  & 's & 0.859276 & 0.001443 & 427 & 8 \\
obl< nsubj> det>  & the & 0.857382 & 0.001518 & 325 & 5 \\
case>  & of

In [111]:
feat_freq_df.loc['topic','case>','by']

count    557
Name: (topic, case>, by), dtype: int64

In [112]:
clust_featdx_df=cresTop['clust_featdx_df']
clust_featdx_df.reset_index(0).sort_index().loc['case>','by']

cluster        Dx    N
arcpath lemma                        
case>   by           0  0.009342  270
        by           1  0.001265   95
        by           4 -0.000515   18
        by           5  0.004931  142
        by           6 -0.000273   27
        by           7 -0.000228    2

In [ ]:
#len(tf_df.loc['topic'].reset_index().set_index(['arcpath','lemma']).loc['case>','by'].token.unique())

True

In [ ]:
from matplotlib import pyplot as plt
#plt.plot(list(bdf.Dx),list(bdf.score_pvalue),'o')
#plt.plot(list(bdf.Dx),list(bdf.cos_pvalue),'*')
#plt.plot(list(bdf.Dx),list(bdf.cos_stat),'.')
#plt.plot(list(bdf.Dx),list(bdf.score_stat),'.')
#plt.plot(list(bdf.score_pvalue),list(bdf.cos_pvalue),'.')
#plt.plot(list(cdf.effect),list(cdf.test_st),'o')
#plt.plot(list(cdf.test_N),list(cdf.test_pvalue),'.')


In [319]:
cresTax['C1']['cl_elong_var']

7.8649476230195035

In [320]:
df   = cresTax['C1']['book_cl_elong']
evar = cresTax['C1']['cl_elong_var']

# Effective variance: effect step^2 * Bernoulli variance. (Var[X] = p*(1-p))

df['p']=df['test_Nx']/df['test_N']
df['np']=1-df['p']
df['eff_var']=df['eff2']*df['p']*df['np'] 
df


effect        Dx   N      eff2  test_pvalue  test_dX  \
arcpath    lemma                                                               
cc>        and       -0.751177  0.002718  13  0.564267      0.03505   -0.631   
nmod>case> of         0.729722  0.004417   8  0.532494      0.20712    0.477   
nmod>      Hewitson   0.725210  0.001158   1  0.525929      0.25272    1.202   
nsubj<     order      0.717932  0.001071   1  0.515427      0.40541    0.875   
nmod>case> in         0.702984  0.005788   9  0.494187      0.48678    0.249   
cc>        as        -0.698172  0.002902   3  0.487444      0.44919   -0.462   
nmod>      Boisduval  0.672279  0.001131   1  0.451959      0.30727    1.073   
nmod>case> with       0.668601  0.002075   4  0.447027      0.32006    0.527   
           as         0.662159  0.000889   1  0.438454      0.30989    1.068   

                      test_Nx  test_N  test_st         p        np   eff_var  
arcpath    lemma                                                              
cc>        and             13     199    -2.12  0.065327  0.934673  0.034454  
nmod>case> of               8     199     1.27  0.040201  0.959799  0.020546  
nmod>      Hewitson         1     199     1.15  0.005025  0.994975  0.002630  
nsubj<     order            1     199     0.83  0.005025  0.994975  0.002577  
nmod>case> in               9     199     0.70  0.045226  0.954774  0.021339  
cc>        as               3     199    -0.76  0.015075  0.984925  0.007238  
nmod>      Boisduval        1     199     1.02  0.005025  0.994975  0.002260  
nmod>case> with             4     199     1.00  0.020101  0.979899  0.008805  
           as               1     199     1.02  0.005025  0.994975  0.002192

In [327]:
def cres_clean(cres):
    
    cdf = pd.concat(
        [cres['C%d'%i]['book_cl_elong'].sort_values(by='test_pvalue').assign(cluster=i) for i in range(10) if 'C%d'%i in cres.keys()]
    ).sort_values('test_pvalue')
    cdf=cdf[cdf.test_pvalue<.1]
    cdf=cdf.groupby('lemma').head(1)
    cdf=cdf.groupby('arcpath').head(2)
    cdf['eff ratio']=(cdf['effect']/cdf['test_dX']).round(2)
    
    # Effective variance: effect step^2 * Bernoulli variance. (Var[X] = p*(1-p))
    
    cdf['p']=cdf['test_Nx']/cdf['test_N']
    cdf['np']=1-cdf['p']
    cdf['eff var']=cdf['eff2']*cdf['p']*cdf['np']     
    
    cdf=cdf[['cluster', 'test_Nx', 'test_N',   'test_pvalue', 'test_st',  'eff var', 'eff ratio']]
    cdf.columns=['cl. id', 'Nfeat', 'cl. size',   't test p-value','t test stat', 'eff var', 'eff ratio']
    cdf.sort_values(by='eff var',ascending=False,inplace=True)
    
    return cdf.head(5).sort_values(by=['cl. id'])
    
    #return pd.concat([df for (c,df) in cdf.head(5).groupby('cl. id')])

In [359]:
sets = {'Adaptation':cresAda,
        'Location':cresLoc,
        'Taxon group':cresTax,
        'Topic':cresTop
       }

for title,cdf in sets.items():
    cdf = cres_clean(cdf)
    print("{\\bf       %s }\n"%title)
    cleanlatex(cdf)
    

    

{\bf       Adaptation }

\begin{tabular}{llrrrrrrr}
\toprule
arcpath & lemma & cl. id & Nfeat & cl. size & t test p-value & t test stat & eff var & eff ratio \\
\midrule
nmod> case>  & of & 0 & 7 & 114 & 0.00 & 5.18 & 0.036141 & 0.48 \\
nmod> case>  & such & 0 & 3 & 114 & 0.002880 & 3.05 & 0.014053 & 0.47 \\
cc>  & or & 6 & 14 & 592 & 0.000020 & 4.35 & 0.007815 & 0.96 \\
amod>  & other & 8 & 8 & 287 & 0.00 & -5.02 & 0.013205 & 0.71 \\
conj>  & feed & 9 & 5 & 284 & 0.000390 & -3.59 & 0.013245 & 0.37 \\
\bottomrule
\end{tabular}

{\bf       Location }

\begin{tabular}{llrrrrrrr}
\toprule
arcpath & lemma & cl. id & Nfeat & cl. size & t test p-value & t test stat & eff var & eff ratio \\
\midrule
det>  & the & 4 & 7 & 13 & 0.088610 & -1.87 & 0.145010 & 0.38 \\
compound<  & locust & 7 & 21 & 298 & 0.00 & 6.24 & 0.011691 & 0.89 \\
conj>  & marsh & 8 & 13 & 197 & 0.00 & 6.63 & 0.050478 & 0.29 \\
conj>  & grassland & 8 & 5 & 197 & 0.000310 & 3.68 & 0.020155 & 0.30 \\
nsubj:pass< aux:pass>  & b

In [340]:
cres_clean(cresAda)


cl. id  Nfeat  cl. size  t test p-value  t test stat  \
arcpath    lemma                                                         
nmod>case> of          0      7       114         0.00000         5.18   
           such        0      3       114         0.00288         3.05   
cc>        or          6     14       592         0.00002         4.35   
amod>      other       8      8       287         0.00000        -5.02   
conj>      feed        9      5       284         0.00039        -3.59   

                   eff var  eff ratio  
arcpath    lemma                       
nmod>case> of     0.036141       0.48  
           such   0.014053       0.47  
cc>        or     0.007815       0.96  
amod>      other  0.013205       0.71  
conj>      feed   0.013245       0.37

In [341]:
cres_clean(cresLoc)


cl. id  Nfeat  cl. size  t test p-value  \
arcpath              lemma                                                
det>                 the             4      7        13         0.08861   
compound<            locust          7     21       298         0.00000   
conj>                marsh           8     13       197         0.00000   
                     grassland       8      5       197         0.00031   
nsubj:pass<aux:pass> be              9      8        36         0.03791   

                                t test stat   eff var  eff ratio  
arcpath              lemma                                        
det>                 the              -1.87  0.145010       0.38  
compound<            locust            6.24  0.011691       0.89  
conj>                marsh             6.63  0.050478       0.29  
                     grassland         3.68  0.020155       0.30  
nsubj:pass<aux:pass> be                2.16  0.010977       0.50

In [342]:
cres_clean(cresTax)


cl. id  Nfeat  cl. size  t test p-value  \
arcpath             lemma                                              
conj>cc>            and           2    277      2280         0.00000   
list<list>compound> Guinean       5      4         5         0.01515   
nmod>case>          in            7     10       128         0.00000   
                    to            7      6       128         0.00022   
det>                a             9     63      2384         0.00002   

                             t test stat   eff var  eff ratio  
arcpath             lemma                                      
conj>cc>            and           -17.13  0.036466       0.52  
list<list>compound> Guinean        -5.03  0.012449       0.53  
nmod>case>          in             -4.82  0.032643       0.18  
                    to             -3.81  0.020633       0.17  
det>                a              -4.23  0.021871       2.15

In [343]:
cres_clean(cresTop)


cl. id  Nfeat  cl. size  t test p-value  t test stat  \
arcpath    lemma                                                           
amod>      large         4     42      8870             0.0         6.09   
           most          4     40      8870             0.0         7.86   
conj>      hare          6     18      2245             0.0        -9.56   
           carrion       6     14      2245             0.0        -9.81   
nmod>case> such          7     26      1889             0.0        -6.39   

                     eff var  eff ratio  
arcpath    lemma                         
amod>      large    0.003613       1.59  
           most     0.003605       1.23  
conj>      hare     0.007340       0.21  
           carrion  0.005482       0.18  
nmod>case> such     0.005787       0.46